<a href="https://colab.research.google.com/github/mani2222278/Rebounce-Applied-AI-and-Analytics-code/blob/main/python_301.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [ ]:
url = "https://huggingface.co/datasets/aarav912/online-retail/resolve/main/online_retail.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


In [ ]:
print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(df.duplicated().sum())

print("\n--- Shape ---")
print(df.shape)


In [ ]:
# Original shape
print("Original shape:", df.shape)

# 1. Missing CustomerID → drop them
# Reason: Without CustomerID we cannot link transactions to customers,
# and a large share of rows are missing it. Dropping is cleaner for analysis.
df = df.dropna(subset=["CustomerID"])
print("After dropping missing CustomerID:", df.shape)

# 2. Convert InvoiceDate to proper datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print("After converting InvoiceDate:", df.shape)

# 3. Remove exact duplicate rows
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

# 4. Remove invalid transactions
# Quantity ≤ 0 or UnitPrice ≤ 0 are cancellations / adjustments, not real sales
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print("After removing invalid transactions:", df.shape)


In [ ]:
df["Sales"] = df["Quantity"] * df["UnitPrice"]
print("After creating Sales column:", df.shape)
df.head()



In [ ]:
sales = df["Sales"].values

mean_sales   = np.mean(sales)
median_sales = np.median(sales)
min_sales    = np.min(sales)
max_sales    = np.max(sales)
std_sales    = np.std(sales)

print("Mean Sales      :", round(mean_sales, 2))
print("Median Sales    :", round(median_sales, 2))
print("Minimum Sales   :", round(min_sales, 2))
print("Maximum Sales   :", round(max_sales, 2))
print("Std Deviation   :", round(std_sales, 2))

# Text cell answer:
# The mean (≈ 22.63) is higher than the median (12.45).
# This gap shows that the Sales distribution is right-skewed —
# a small number of very large wholesale orders are pulling the average up.



In [ ]:
# 1. How many transactions have Sales above 500?
above_500 = df[df["Sales"] > 500]
print("Transactions with Sales > 500:", len(above_500))

# 2. What are the 10 largest transactions by Sales?
top10 = df.sort_values("Sales", ascending=False).head(10)
print("\nTop 10 largest transactions:")
print(top10[["InvoiceNo", "Description", "Quantity", "UnitPrice", "Sales", "Country"]])

# 3. How many transactions come from Germany?
germany = df[df["Country"] == "Germany"]
print("\nTransactions from Germany:", len(germany))

# 4. Which transactions have Quantity above 1,000?
qty_1000 = df[df["Quantity"] > 1000]
print("\nTransactions with Quantity > 1000:", len(qty_1000))
print(qty_1000[["InvoiceNo", "Description", "Quantity", "UnitPrice", "Sales", "Country"]].head(10))

# Observation: Many of these large quantities look believable because
# the company serves wholesalers. Extremely large single-item orders
# (e.g. thousands of the same product) are common for bulk buyers.


In [ ]:
summary = (
    df.groupby("Country")
      .agg(
          Number_of_Transactions=("Sales", "count"),
          Total_Sales=("Sales", "sum"),
          Average_Sales=("Sales", "mean")
      )
      .reset_index()
      .sort_values("Total_Sales", ascending=False)
)

print(summary)

print("\n--- Top 5 countries by Total Sales ---")
print(summary.head(5))

print("\n--- Country with highest Average Sales ---")
print(summary.sort_values("Average_Sales", ascending=False).head(3))

In [ ]:
region_lookup = {
    "United Kingdom": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Spain": "Europe",
    "Netherlands": "Europe",
    "Belgium": "Europe",
    "Switzerland": "Europe",
    "Portugal": "Europe",
    "Australia": "Oceania",
    "Japan": "Asia",
    "USA": "Americas",
}

region_df = pd.DataFrame(list(region_lookup.items()), columns=["Country", "Region"])

# Left merge
merged = summary.merge(region_df, on="Country", how="left")
merged["Region"] = merged["Region"].fillna("Other")

print(merged.head(15))

# Total sales per region
region_summary = (
    merged.groupby("Region")["Total_Sales"]
          .sum()
          .sort_values(ascending=False)
)
print("\n--- Total Sales by Region ---")
print(region_summary)

In [ ]:
# 1. Which country generated the highest total sales?
# United Kingdom with approximately £7,285,025 total sales.

# 2. Which country had the highest average transaction value?
# Netherlands with an average of about £121 per transaction.

# 3. Which region performed best?
# Europe generated the highest total sales (≈ £8.20 million).

# 4. What is one interesting pattern you noticed in the data?
# The country with the highest total sales (United Kingdom) is not the one
# with the highest average transaction value. Smaller countries like
# Netherlands, Australia and Japan show much higher average order values,
# suggesting a higher proportion of wholesale buyers.

# 5. What is one data-quality problem you ran into, and how did you handle it?
# CustomerID was missing in about 135,000 rows. I decided to drop those rows
# because without a CustomerID the transaction cannot be linked to a customer
# and would distort customer-level analysis. I also removed negative and zero
# Quantity/UnitPrice rows as they represent cancellations rather than sales.



In [ ]:
# Final Insight
#
# The cleaned Online Retail dataset shows that the United Kingdom dominates
# total sales, contributing more than £7.2 million. However, the average
# transaction value is highest in the Netherlands (£121), followed by
# Australia and Japan. This comparison reveals that while the UK generates
# volume, several smaller markets generate higher-value wholesale orders.
#
# The Sales distribution is right-skewed: the mean (£22.63) sits well above
# the median (£12.45), confirming that a small number of very large orders
# pull the average upward.
#
# One limitation is that we dropped all rows with missing CustomerID and
# all cancellation/refund rows. The final numbers therefore reflect only
# completed sales linked to known customers, and may understate the true
# scale of the business if many guest checkouts occurred